# Phase 3 — Market Rotation Mechanism Study
GitHub → tests → FinLab → Phase 3 → immutable Google Drive archive.

C0 outcomes are mechanism anchors and are not tradable entry returns.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil, subprocess
from pathlib import Path
from google.colab import userdata

REPO_URL = 'https://github.com/hh4832/-institutional-spot-flow-study.git'
BRANCH = 'phase3-market-rotation'
PROJECT_DIR = Path('/content/institutional-spot-flow-study')
DRIVE_FOLDER_ID = '1zjTMbjv-SiDhkeiIpUaidZqTYGDalold'
DRIVE_OUTPUT_ROOT = Path('/content/drive/MyDrive/Quant_Research/institutional-spot-flow-study/phase3_outputs')
# Public clone requires no GitHub token.
os.chdir('/content')
if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)
result = subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(PROJECT_DIR)], text=True, capture_output=True)
print(result.stdout)
print(result.stderr)
if result.returncode != 0:
    raise RuntimeError(f'git clone failed with exit code {result.returncode}\n{result.stderr}')
os.chdir(PROJECT_DIR)
subprocess.run(['git', 'remote', '-v'], check=True)
subprocess.run(['git', 'branch', '--show-current'], check=True)
subprocess.run(['git', 'status', '--short', '--branch'], check=True)
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('Git commit:', commit)

In [ ]:
subprocess.run(['python', '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
tests = subprocess.run(['python', '-m', 'pytest', '-q'])
if tests.returncode != 0:
    raise RuntimeError('Tests failed; stopping Phase 3 run.')

In [ ]:
import finlab
try:
    finlab_token = userdata.get('FINLAB_API_TOKEN')
except Exception:
    finlab_token = None
finlab.login(finlab_token) if finlab_token else finlab.login()
del finlab_token

In [ ]:
from config import StudyConfig
from data_loader import load_finlab_data
from phase3_pipeline import run_phase3_study

raw = load_finlab_data(ticker='0050', include_otc_indices=True)
config = StudyConfig(study_mode='phase3_rotation', output_root=PROJECT_DIR / 'outputs_phase3')
output_dir = run_phase3_study(raw, config, tests_passed=True)
print('Local output:', output_dir)

In [ ]:
DRIVE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
destination = DRIVE_OUTPUT_ROOT / output_dir.name
if destination.exists():
    raise FileExistsError(f'Destination exists; refusing overwrite: {destination}')
shutil.copytree(output_dir, destination)
print('Drive folder ID:', DRIVE_FOLDER_ID)
print('Archived to:', destination)

In [ ]:
required = {'phase3_run_metadata.json', 'phase3_config_snapshot.json', 'phase3_otc_index_audit.csv', 'phase3_candidate_signals.csv', 'phase3_outcome_dataset.parquet', 'phase3_absolute_returns.csv', 'phase3_relative_rotation_results.csv', 'phase3_primary_regressions.csv', 'phase3_significant_results.csv', 'phase3_total_return_results.csv', 'phase3_price_index_sensitivity.csv', 'phase3_temporal_robustness.csv', 'phase3_signal_comparison.csv', 'phase3_summary.md', 'run_info.txt'}
missing = required - {p.name for p in destination.iterdir()}
if missing:
    raise RuntimeError(f'Drive archive missing files: {sorted(missing)}')
print('Phase 3 archive verified:', destination)

In [ ]:
import pandas as pd
display(pd.read_csv(destination / 'phase3_signal_comparison.csv').head(30))
print((destination / 'phase3_summary.md').read_text())